In [1]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

In [3]:
class MultiTaskUNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        
        self.enc1 = DoubleConv(1, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        
        self.bottleneck = DoubleConv(512, 1024)

        
        self.upconv4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.segmentation_head = nn.Conv2d(64, 1, kernel_size=1)  # binary mask

        # Classification Head
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
       
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

       
        b = self.bottleneck(self.pool4(e4))

        
        d4 = self.upconv4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))
        d3 = self.upconv3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.upconv2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.upconv1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        seg_out = torch.sigmoid(self.segmentation_head(d1))  # binary mask (0–1)

        # Classification Head
        pooled = self.global_pool(b)
        cls_out = self.classifier(pooled)  # logits (will apply softmax in loss)

        return seg_out, cls_out

In [4]:
label_encoder = {'normal': 0, 'benign': 1, 'malignant': 2}


In [5]:
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiTaskUNet(num_classes=3)
model.load_state_dict(torch.load("/kaggle/input/trained-model/pytorch/default/1/best_model.pth"))
model.to(device)
model.eval()


MultiTaskUNet(
  (enc1): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (enc2): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
 

In [7]:
# import matplotlib.pyplot as plt
# import glob

# def predict_and_visualize_from_folder(folder_path):
#     image_paths = glob.glob(os.path.join(folder_path, "*.png")) + \
#                   glob.glob(os.path.join(folder_path, "*.jpg"))

#     for image_path in sorted(image_paths):
#         image = Image.open(image_path).convert('L')
#         input_tensor = transform(image).unsqueeze(0).to(device)

#         with torch.no_grad():
#             pred_mask, class_logits = model(input_tensor)

#         pred_mask = pred_mask.squeeze().cpu().numpy()
#         class_id = class_logits.argmax(dim=1).item()
#         class_name = list(label_encoder.keys())[class_id]


#         # Visualization
#         image_np = np.array(image.resize((256, 256)))
#         plt.figure(figsize=(12, 4))

#         plt.subplot(1, 3, 1)
#         plt.imshow(image_np, cmap='gray')
#         plt.title("Input Image")
#         plt.axis('off')

#         plt.subplot(1, 3, 2)
#         plt.imshow(pred_mask, cmap='gray')
#         plt.title("Predicted Mask")
#         plt.axis('off')

#         plt.subplot(1, 3, 3)
#         plt.imshow(image_np, cmap='gray')
#         plt.imshow(pred_mask, cmap='Reds', alpha=0.4)
#         plt.title(f"Overlay - Class: {class_name}")
#         plt.axis('off')

#         plt.suptitle(f"File: {os.path.basename(image_path)}", fontsize=12)
#         plt.tight_layout()
#         plt.show()

In [8]:
# predict_and_visualize_from_folder("/kaggle/input/test-case/test_data")

In [9]:
# import glob
# import os

# def load_image_mask_pairs(folder_path):
#     mask_paths = glob.glob(os.path.join(folder_path,"*_mask.png"))
#     image_mask_pairs = []

#     for mask_path in mask_paths:
#         image_path = mask_path.replace("_mask", "")
#         if os.path.exists(image_path):
#             image_mask_pairs.append((image_path, mask_path))

#     return image_mask_pairs



In [10]:
# from PIL import Image
# import matplotlib.pyplot as plt
# import numpy as np

# def visualize_predictions_on_test_set(image_mask_pairs):
#     model.eval()
#     for img_path, mask_path in image_mask_pairs:
#         image = Image.open(img_path).convert('L')
#         true_mask = Image.open(mask_path).convert('L')

#         input_tensor = transform(image).unsqueeze(0).to(device)

#         with torch.no_grad():
#             pred_mask, class_logits = model(input_tensor)

#         pred_mask_np = pred_mask.squeeze().cpu().numpy()
#         true_mask_np = np.array(true_mask.resize((256, 256))) / 255.0
#         image_np = np.array(image.resize((256, 256)))

#         class_id = class_logits.argmax(dim=1).item()
#         class_name = list(label_encoder.keys())[class_id]

#         # Show results
#         plt.figure(figsize=(15, 4))

#         plt.subplot(1, 4, 1)
#         plt.imshow(image_np, cmap='gray')
#         plt.title("Input Image")
#         plt.axis('off')

#         plt.subplot(1, 4, 2)
#         plt.imshow(true_mask_np, cmap='gray')
#         plt.title("Ground Truth Mask")
#         plt.axis('off')

#         plt.subplot(1, 4, 3)
#         plt.imshow(pred_mask_np, cmap='gray')
#         plt.title("Predicted Mask")
#         plt.axis('off')

#         plt.subplot(1, 4, 4)
#         plt.imshow(image_np, cmap='gray')
#         plt.imshow(pred_mask_np, cmap='Reds', alpha=0.4)
#         plt.title(f"Overlay\nPredicted Class: {class_name}")
#         plt.axis('off')

#         plt.tight_layout()
#         plt.show()


In [11]:
# test_pairs = load_image_mask_pairs("/kaggle/input/test-data-2/test_data_2")
# visualize_predictions_on_test_set(test_pairs)
